# **Generative AI - Text Generation and Machine Translation | Assignment**

Question 1: What is Generative AI and what are its primary use cases across
industries?
- Generative AI is a subset of artificial intelligence that focuses on creating new content—such as text, images, audio, video, and code—by learning patterns from existing data using advanced models like neural networks and transformer architectures. Unlike traditional AI, which mainly analyzes or predicts, generative AI can produce original outputs that resemble human-created work. Its primary use cases span multiple industries: in healthcare, it helps generate medical reports and drug discovery insights; in marketing, it creates personalized content and advertisements; in software development, it assists with code generation and debugging; in entertainment, it produces music, art, and scripts; in finance, it supports risk analysis and report automation; and in education, it enables personalized learning materials and tutoring systems.

Question 2: Explain the role of probabilistic modeling in generative models. How do
these models differ from discriminative models?
- Probabilistic modeling is central to generative models because it focuses on learning the underlying probability distribution of data, typically expressed as ( P(X) ) or the joint distribution ( P(X, Y) ), allowing the model to generate new, realistic data samples similar to the training data (e.g., images, text, or audio). Models like Generative Adversarial Networks and Variational Autoencoders use probabilistic approaches to capture data patterns and variability. In contrast, discriminative models aim to learn the conditional probability ( P(Y \mid X) ), directly mapping inputs to outputs for tasks like classification or regression, without modeling how the data is generated. Thus, generative models can both understand and create data, while discriminative models focus only on decision boundaries and prediction accuracy.

Question 3: What is the difference between Autoencoders and Variational
Autoencoders (VAEs) in the context of text generation?
- Autoencoders and Variational Autoencoders (VAEs) differ mainly in how they represent and generate data. A standard autoencoder learns a deterministic mapping that compresses input text into a fixed latent vector and then reconstructs the same text, making it useful for tasks like denoising or feature extraction but not ideal for generating new, diverse text. In contrast, a Variational Autoencoder introduces a probabilistic approach, where the latent space is modeled as a distribution (typically Gaussian), allowing the model to sample different latent vectors and generate varied, coherent text outputs. This makes VAEs better suited for text generation tasks, as they encourage smoother latent spaces and enable creativity and diversity in generated sentences, whereas traditional autoencoders tend to reproduce inputs rather than generate new ones.

Question 4: Describe the working of attention mechanisms in Neural Machine
Translation (NMT). Why are they critical?
-In Neural Machine Translation (NMT), attention mechanisms allow the model to dynamically focus on different parts of the source sentence while generating each word of the target sentence. Instead of relying on a single fixed-length context vector (as in early encoder–decoder models), attention computes alignment scores between the current decoder state and all encoder hidden states, assigning higher weights to the most relevant words in the input. This produces a context vector that changes at every decoding step, enabling the model to capture long-range dependencies and word alignments more effectively. Attention is critical because it significantly improves translation quality, especially for long or complex sentences, reduces information loss, and provides interpretability by showing which source words the model is focusing on during translation.

Question 5: What ethical considerations must be addressed when using generative AI
for creative content such as poetry or storytelling?
- When using generative AI for creative content like poetry or storytelling, several ethical considerations must be addressed, including originality and plagiarism (ensuring the AI does not replicate copyrighted or existing works), proper attribution and transparency (clearly disclosing AI involvement), bias and representation (avoiding harmful stereotypes or exclusion in generated content), and respect for intellectual property rights of training data sources. Additionally, there are concerns about the impact on human creators, such as fair compensation and potential job displacement, as well as the responsible use of AI to prevent misinformation, deepfakes, or harmful narratives. Ensuring accountability, consent where applicable, and maintaining creative integrity are essential for ethical AI use in artistic domains.



Question 6: Use the following small text dataset to train a simple Variational
Autoencoder (VAE) for text reconstruction:
["The sky is blue", "The sun is bright", "The grass is green",
"The night is dark", "The stars are shining"]
1. Preprocess the data (tokenize and pad the sequences).
2. Build a basic VAE model for text reconstruction.
3. Train the model and show how it reconstructs or generates similar sentences.
Include your code, explanation, and sample outputs.
(Include your Python code and output in the code box below.)


In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Lambda, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

# --------------------------
# 1. Dataset
# --------------------------
texts = [
    "The sky is blue",
    "The sun is bright",
    "The grass is green",
    "The night is dark",
    "The stars are shining"
]

# --------------------------
# 2. Preprocessing
# --------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
max_len = max(len(seq) for seq in sequences)
vocab_size = len(tokenizer.word_index) + 1

padded = pad_sequences(sequences, maxlen=max_len, padding='post')

# --------------------------
# 3. VAE Model
# --------------------------
embedding_dim = 16
latent_dim = 8

# Sampling function
def sampling(args):
    z_mean, z_log_var = args
    epsilon = K.random_normal(shape=(K.shape(z_mean)[0], latent_dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

class VAE(Model):
    def __init__(self, vocab_size, embedding_dim, latent_dim, max_len):
        super(VAE, self).__init__()
        self.embedding_layer = Embedding(vocab_size, embedding_dim)
        self.encoder_lstm = LSTM(16)
        self.z_mean_dense = Dense(latent_dim)
        self.z_log_var_dense = Dense(latent_dim)
        self.decoder_dense_1 = Dense(16, activation='relu')
        self.decoder_output_dense = Dense(max_len * vocab_size)
        self.decoder_reshape = Reshape((max_len, vocab_size))
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.latent_dim = latent_dim

    def encode(self, inputs):
        x = self.embedding_layer(inputs)
        x = self.encoder_lstm(x)
        z_mean = self.z_mean_dense(x)
        z_log_var = self.z_log_var_dense(x)
        return z_mean, z_log_var

    def decode(self, z):
        decoder_inputs = self.decoder_dense_1(z)
        decoder_outputs = self.decoder_output_dense(decoder_inputs)
        decoder_outputs = self.decoder_reshape(decoder_outputs)
        return decoder_outputs

    def call(self, inputs):
        z_mean, z_log_var = self.encode(inputs)
        z = Lambda(sampling, output_shape=(self.latent_dim,))([z_mean, z_log_var])
        decoder_outputs = self.decode(z)

        # VAE Loss
        # Calculate reconstruction loss for each token, then sum per sequence, then mean over batch
        reconstruction_loss_per_token = tf.keras.losses.sparse_categorical_crossentropy(
            inputs, decoder_outputs, from_logits=True
        )
        reconstruction_loss = tf.reduce_mean(tf.reduce_sum(reconstruction_loss_per_token, axis=-1))

        # KL loss: sum over latent dimensions per sample, then mean over batch
        kl_loss = -0.5 * K.mean(K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1))

        total_loss = reconstruction_loss + kl_loss
        self.add_loss(total_loss)

        return decoder_outputs

# Instantiate and compile the VAE model
vae = VAE(vocab_size, embedding_dim, latent_dim, max_len)
vae.compile(optimizer='adam')

# --------------------------
# 4. Training
# --------------------------
# When using add_loss, the model implicitly knows its targets
vae.fit(x=padded, epochs=200, verbose=0)

# --------------------------
# 5. Reconstruction
# --------------------------
preds = vae.predict(padded)

# Convert predictions back to words
index_word = {v: k for k, v in tokenizer.word_index.items()}

def decode_sequence(seq):
    words = []
    for idx in seq:
        if idx == 0:
            continue
        words.append(index_word.get(idx, ""))
    return " ".join(words)

# Get predicted words
predicted_sentences = []
for pred in preds:
    # Apply softmax manually for argmax to get predicted word IDs from logits
    pred_probs = tf.nn.softmax(pred, axis=-1).numpy()
    pred_ids = np.argmax(pred_probs, axis=1)
    predicted_sentences.append(decode_sequence(pred_ids))

# --------------------------
# Output
# --------------------------
print("\nOriginal vs Reconstructed:\n")
for i in range(len(texts)):
    print(f"Original: {texts[i]}")
    print(f"Reconstructed: {predicted_sentences[i]}\n")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step

Original vs Reconstructed:

Original: The sky is blue
Reconstructed: the night is shining

Original: The sun is bright
Reconstructed: the grass is bright

Original: The grass is green
Reconstructed: the stars is dark

Original: The night is dark
Reconstructed: the stars is dark

Original: The stars are shining
Reconstructed: the sky is dark



Question 7: Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short
English paragraph into French and German. Provide the original and translated text.


In [ ]:
from openai import OpenAI
import os
from google.colab import userdata

# Ensure your OpenAI API key is set as an environment variable (recommended)
# or pass it directly: client = OpenAI(api_key="YOUR_OPENAI_API_KEY")
# For Colab, you can use: from google.colab import userdata; client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
# For this example, we assume OPENAI_API_KEY is set in your environment or Colab secrets.
client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY') # Retrieve API key from Colab secrets
)

text = "Artificial intelligence is transforming the world rapidly."

prompt_content = f"Translate the following text into French and German:\n{text}"

response = client.chat.completions.create(
    model="gpt-4o-mini", # Corrected model name
    messages=[
        {"role": "user", "content": prompt_content}
    ]
)

print(response.choices[0].message.content)


Question 8: Implement a simple attention-based encoder-decoder model for
English-to-Spanish translation using Tensorflow or PyTorch

In [17]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --------------------------
# 1. Sample Dataset
# --------------------------
eng_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "thank you",
    "good night"
]

spa_sentences = [
    "hola",
    "como estas",
    "estoy bien",
    "gracias",
    "buenas noches"
]

# --------------------------
# 2. Tokenization
# --------------------------
eng_tokenizer = Tokenizer()
spa_tokenizer = Tokenizer(filters='')

eng_tokenizer.fit_on_texts(eng_sentences)
spa_tokenizer.fit_on_texts(["<start> " + s + " <end>" for s in spa_sentences])

eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)
spa_seq = spa_tokenizer.texts_to_sequences(["<start> " + s + " <end>" for s in spa_sentences])

max_len_eng = max(len(s) for s in eng_seq)
max_len_spa = max(len(s) for s in spa_seq)

eng_seq = pad_sequences(eng_seq, maxlen=max_len_eng, padding='post')
spa_seq = pad_sequences(spa_seq, maxlen=max_len_spa, padding='post')

vocab_eng = len(eng_tokenizer.word_index) + 1
vocab_spa = len(spa_tokenizer.word_index) + 1

# --------------------------
# 3. Encoder
# --------------------------
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        output, h, c = self.lstm(x)
        return output, h, c

# --------------------------
# 4. Attention (Bahdanau)
# --------------------------
class Attention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, hidden, enc_output):
        hidden = tf.expand_dims(hidden, 1)
        score = self.V(tf.nn.tanh(self.W1(enc_output) + self.W2(hidden)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * enc_output
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# --------------------------
# 5. Decoder
# --------------------------
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = Attention(units)

    def call(self, x, hidden, cell, enc_output):
        context_vector, attn = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, h, c = self.lstm(x, initial_state=[hidden, cell])
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, h, c, attn

# --------------------------
# 6. Initialize Model
# --------------------------
embedding_dim = 64
units = 128

encoder = Encoder(vocab_eng, embedding_dim, units)
decoder = Decoder(vocab_spa, embedding_dim, units)

optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# --------------------------
# 7. Training Step
# --------------------------
@tf.function
def train_step(inp, targ):
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_h, enc_c = encoder(inp)

        dec_input = tf.expand_dims([spa_tokenizer.word_index['<start>']] * inp.shape[0], 1)

        dec_h, dec_c = enc_h, enc_c

        for t in range(1, targ.shape[1]):
            predictions, dec_h, dec_c, _ = decoder(dec_input, dec_h, dec_c, enc_output)
            loss += loss_object(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1)

    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return loss

# --------------------------
# 8. Training Loop
# --------------------------
EPOCHS = 200

for epoch in range(EPOCHS):
    loss = train_step(eng_seq, spa_seq)
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.numpy():.4f}")

# --------------------------
# 9. Inference (Translation)
# --------------------------
def translate(sentence):
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_len_eng, padding='post')

    enc_out, enc_h, enc_c = encoder(seq)

    dec_input = tf.expand_dims([spa_tokenizer.word_index['<start>']], 0)
    dec_h, dec_c = enc_h, enc_c

    result = []

    for _ in range(max_len_spa):
        predictions, dec_h, dec_c, _ = decoder(dec_input, dec_h, dec_c, enc_out)
        pred_id = tf.argmax(predictions[0]).numpy()

        word = spa_tokenizer.index_word.get(pred_id, '')
        if word == "<end>":
            break

        result.append(word)
        dec_input = tf.expand_dims([pred_id], 0)

    return " ".join(result)

# --------------------------
# 10. Test Output
# --------------------------
print("\nTranslations:\n")
for s in eng_sentences:
    print(f"{s} -> {translate(s)}")

Epoch 0, Loss: 7.1887
Epoch 50, Loss: 1.8503
Epoch 100, Loss: 0.0469
Epoch 150, Loss: 0.0161

Translations:

hello -> hola
how are you -> como estas
i am fine -> estoy bien
thank you -> gracias
good night -> buenas noches


Question 9: Use the following short poetry dataset to simulate poem generation with a
pre-trained GPT model:
["Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."]
Using this dataset as a reference for poetic structure and language, generate a new 2-4
line poem using a pre-trained GPT model (such as GPT-2). You may simulate
fine-tuning by prompting the model with similar poetic patterns.
Include your code, the prompt used, and the generated poem in your answer.



In [18]:
from transformers import pipeline

# Load GPT-2 text generator
generator = pipeline("text-generation", model="gpt2")

# Prompt (simulated fine-tuning)
prompt = """
Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

Write a new short poem (2-4 lines) in a similar style:
"""

# Generate poem
output = generator(prompt, max_length=80, num_return_sequences=1)

# Print result
print("Generated Poem:\n")
print(output[0]['generated_text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Poem:


Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

Write a new short poem (2-4 lines) in a similar style:

The moon glows bright in silent skies,

A bird sings where the soft wind sighs.

Write a new short poem (5-6 lines) in a similar style:

The moon glows bright in silent skies,

A bird sings where the soft wind sighs.

Write a new short poem (7-8 lines) in a similar style:

The moon glows bright in silent skies,

A bird sings where the soft wind sighs.

Write a new short poem (9-10 lines) in a similar style:

The moon glows bright in silent skies,

A bird sings where the soft wind sighs.

Write a new short poem (11-12 lines) in a similar style:

The moon glows bright in silent skies,

A bird sings where the soft wind sighs.

Write a new short poem (13-14 lines) in a similar style:

The moon glows bright in silent skies,

A bird sings where the soft wind sighs.

Write a ne

Question 10: Imagine you are building a creative writing assistant for a publishing
company. The assistant should generate story plots and character descriptions using
Generative AI. Describe how you would design the system, including model selection,
training data, bias mitigation, and evaluation methods. Explain the real-world challenges
you might face.
- Designing a creative writing assistant for a publishing company involves combining strong language models with careful data and ethical practices.

First, for **model selection**, I would use a powerful pre-trained generative model such as GPT-4 or similar large language models because they can generate high-quality narratives, plots, and character descriptions with minimal fine-tuning. For more control over tone and genre (e.g., fantasy, romance, thriller), I would apply **prompt engineering** and possibly fine-tune smaller versions like GPT-2 on domain-specific data.

Second, for **training data**, I would curate a diverse dataset of books, scripts, and storytelling resources across genres, ensuring proper licensing and copyright compliance. The dataset should include structured elements such as plot summaries, character arcs, and dialogues to help the model learn storytelling patterns like conflict, climax, and resolution.

Third, **bias mitigation** is critical. Since training data may contain cultural, gender, or racial biases, I would implement filtering pipelines, balanced datasets, and post-generation moderation layers to detect and correct biased or harmful outputs. Human review loops and reinforcement learning with feedback can further improve fairness and inclusivity.

Fourth, for **evaluation methods**, I would combine:

* **Automatic metrics** (e.g., perplexity, diversity scores)
* **Human evaluation** (editors rating creativity, coherence, originality)
* **A/B testing** (comparing different prompts or models in real workflows)

Finally, several **real-world challenges** may arise. These include maintaining originality (avoiding plagiarism), ensuring consistent long-form storytelling, handling ambiguous or vague prompts, and aligning outputs with a publisher’s style guidelines. There are also concerns about over-reliance on AI by writers, data privacy issues, and the computational cost of running large models at scale. Addressing these challenges requires a hybrid approach combining AI capabilities with human creativity and oversight.
